# Optimal Bidding under Single-Price Imbalance Settlement

**Replication notebook** for the paper:
> *Optimal bidding under single-price imbalance settlement in electricity markets: Analytical solutions and distributionally robust strategies*

This notebook reproduces all tables and figures in the paper.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

# Ensure the repo root is on the Python path
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_classes import MarketParameters, OptimizationResult
from src.data_acquisition import load_balancing_data, create_synthetic_balancing_data
from src.data_processing import (
    estimate_parameters, prepare_market_data, clean_data,
    test_normality, summary_statistics, split_train_test,
)
from src.optimization import BalancingMarketOptimizer
from src.dro_solver import optimize_wasserstein_dro, run_dro_sensitivity
from src.backtesting import (
    rolling_backtest, block_bootstrap_pb_star, chronological_backtest,
)
from src.visualization import (
    apply_style, plot_conditional_linearity, plot_gaussianity_check,
    plot_data_overview, plot_backtest_comparison,
)

# Directories
RESULTS_DIR = REPO_ROOT / 'results'
FIG_DIR     = RESULTS_DIR / 'figures'
TAB_DIR     = RESULTS_DIR / 'tables'
for d in [RESULTS_DIR, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

apply_style()
print('✓ Setup complete')

## 1. Data Acquisition

Load Nordic mFRR (DK2) and German aFRR (DE-LU) data. Uses cached `.pkl` files
if available; otherwise fetches from the API or falls back to synthetic data.

In [ ]:
# ── Load both markets ─────────────────────────────────────────────────
data_nordic = load_balancing_data('nordic')
data_german = load_balancing_data('german')

# Clean
data_nordic = clean_data(data_nordic)
data_german = clean_data(data_german)

# Add features
data_nordic = prepare_market_data(data_nordic, 'Nordic mFRR (DK2)')
data_german = prepare_market_data(data_german, 'German aFRR (DE-LU)')

print(f'Nordic: {len(data_nordic):,} hours')
print(f'German: {len(data_german):,} hours')

## 2. Parameter Estimation & Descriptive Statistics

In [ ]:
# ── Estimate bivariate parameters ────────────────────────────────────
params_nordic = estimate_parameters(data_nordic, 'Nordic mFRR (DK2)')
params_german = estimate_parameters(data_german, 'German aFRR (DE-LU)')

print(params_nordic)
print()
print(params_german)

In [ ]:
# ── Descriptive statistics table (Table 1 in paper) ──────────────────
stats_nordic = summary_statistics(data_nordic)
stats_german = summary_statistics(data_german)

display(stats_nordic.round(2))
display(stats_german.round(2))

In [ ]:
# ── Normality tests ──────────────────────────────────────────────────
for name, df in [('Nordic', data_nordic), ('German', data_german)]:
    results = test_normality(df)
    print(f'\n{name}:')
    for series, r in results.items():
        print(f'  {series}: skew={r["skewness"]:.2f}, kurt={r["excess_kurtosis"]:.2f}, '
              f'JB p={r["jarque_bera_p"]:.2e}, reject={r["reject_5pct"]}')

## 3. Figures: Data Overview & Diagnostics

In [ ]:
# ── Figure: Data overview (both markets) ─────────────────────────────
plot_data_overview(data_nordic, params_nordic, FIG_DIR / 'data_visualization_pub_v2_nordic.png')
plot_data_overview(data_german, params_german, FIG_DIR / 'data_visualization_pub_v2_german.png')
print('✓ Data overview figures saved')

In [ ]:
# ── Figure: Conditional linearity diagnostic ─────────────────────────
plot_conditional_linearity(data_nordic, params_nordic, FIG_DIR / 'conditional_linearity.png')
plot_conditional_linearity(data_german, params_german, FIG_DIR / 'conditional_linearity_german.png')
print('✓ Conditional linearity figures saved')

In [ ]:
# ── Figure: Gaussianity check ────────────────────────────────────────
plot_gaussianity_check(data_nordic, params_nordic, FIG_DIR / 'gaussianity_check.png')
plot_gaussianity_check(data_german, params_german, FIG_DIR / 'gaussianity_check_german.png')
print('✓ Gaussianity check figures saved')

## 4. Optimal Bidding (Theorem 4)

In [ ]:
# ── Analytical + empirical optimisation ──────────────────────────────
for name, params, df in [
    ('Nordic mFRR (DK2)',   params_nordic, data_nordic),
    ('German aFRR (DE-LU)', params_german, data_german),
]:
    opt = BalancingMarketOptimizer(params, q=1.0, data=df)
    res_a = opt.optimize_analytical()
    res_e = opt.optimize_empirical()
    is_foc, gap = opt.verify_optimality_condition(res_a.p_bid_optimal)
    
    print(f'\n{name}:')
    print(f'  Analytical:  p^{{b*}} = {res_a.p_bid_optimal:.2f},  E[V] = {res_a.expected_revenue:.2f}')
    print(f'  Empirical:   p^{{b*}} = {res_e.p_bid_optimal:.2f},  E[V] = {res_e.expected_revenue:.2f}')
    print(f'  FOC gap: {gap:.4f} ({"✓" if is_foc else "✗"})')
    print(f'  β = {params.beta:.3f},  P(accept) = {opt.acceptance_probability(res_a.p_bid_optimal):.3f}')

## 5. Block Bootstrap Confidence Intervals

In [ ]:
# ── Bootstrap p^{b*} (both markets) ─────────────────────────────────
pb_boot_nordic, ci_nordic = block_bootstrap_pb_star(data_nordic, B=500, block_len=168, seed=42)
pb_boot_german, ci_german = block_bootstrap_pb_star(data_german, B=500, block_len=168, seed=42)

print(f'Nordic p^{{b*}} 95% CI: [{ci_nordic[0]:.1f}, {ci_nordic[1]:.1f}]')
print(f'German p^{{b*}} 95% CI: [{ci_german[0]:.1f}, {ci_german[1]:.1f}]')

In [ ]:
# ── Figure: Bootstrap distributions ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, samples, ci, name, color in [
    (axes[0], pb_boot_nordic, ci_nordic, 'Nordic mFRR (DK2)', '#2196F3'),
    (axes[1], pb_boot_german, ci_german, 'German aFRR (DE-LU)', '#FF9800'),
]:
    ax.hist(samples, bins=40, density=True, alpha=0.7, color=color, edgecolor='black', lw=0.3)
    ax.axvline(np.mean(samples), color='black', ls='-', lw=2, label=f'Mean={np.mean(samples):.1f}')
    ax.axvline(ci[0], color='red', ls='--', lw=1.5, label=f'95% CI: [{ci[0]:.1f}, {ci[1]:.1f}]')
    ax.axvline(ci[1], color='red', ls='--', lw=1.5)
    ax.set_xlabel('$p^{b*}$ (€/MWh)')
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'bootstrap_pb_star_both.pdf', bbox_inches='tight')
plt.show()
print('✓ Bootstrap figure saved')

## 6. Out-of-Sample Backtesting

In [ ]:
# ── Chronological 70/30 backtest (both markets) ─────────────────────
bt_nordic = chronological_backtest(data_nordic, train_ratio=0.70)
bt_german = chronological_backtest(data_german, train_ratio=0.70)

print('Nordic backtest:')
display(bt_nordic.round(3))
print('\nGerman backtest:')
display(bt_german.round(3))

## 7. Wasserstein DRO

In [ ]:
# ── DRO sensitivity (both markets) ──────────────────────────────────
train_nordic, _ = split_train_test(data_nordic, 0.70)
train_german, _ = split_train_test(data_german, 0.70)

dro_nordic = run_dro_sensitivity(train_nordic, epsilon_grid=[0, 0.25, 0.5, 1.0, 2.0], tau=2.0)
dro_german = run_dro_sensitivity(train_german, epsilon_grid=[0, 0.25, 0.5, 1.0, 2.0], tau=2.0)

print('Nordic DRO:')
display(dro_nordic[['epsilon', 'p_bid_optimal_dro', 'robust_lower_bound',
                     'empirical_mean_smoothed', 'robust_cost_pct']].round(3))
print('\nGerman DRO:')
display(dro_german[['epsilon', 'p_bid_optimal_dro', 'robust_lower_bound',
                     'empirical_mean_smoothed', 'robust_cost_pct']].round(3))

## 8. Sensitivity Analysis

In [ ]:
# ── Sensitivity to ρ ─────────────────────────────────────────────────
opt_nordic = BalancingMarketOptimizer(params_nordic)
rho_range = np.linspace(0.3, 0.98, 20)
sens_df = opt_nordic.sensitivity_analysis('rho', rho_range)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(sens_df['rho'], sens_df['p_bid_optimal'], 'b-o', ms=4)
axes[0].set_xlabel(r'$\rho$'); axes[0].set_ylabel('$p^{b*}$ (€/MWh)')
axes[0].set_title('Optimal bid vs correlation')

axes[1].plot(sens_df['rho'], sens_df['expected_revenue'], 'r-s', ms=4)
axes[1].set_xlabel(r'$\rho$'); axes[1].set_ylabel('E[V] (€/MWh)')
axes[1].set_title('Expected revenue vs correlation')

axes[2].plot(sens_df['rho'], sens_df['acceptance_prob'], 'g-d', ms=4)
axes[2].set_xlabel(r'$\rho$'); axes[2].set_ylabel('P(accept)')
axes[2].set_title('Acceptance probability vs correlation')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'sensitivity.pdf', bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print('=' * 70)
print('REPLICATION SUMMARY')
print('=' * 70)
for name, params, bt, ci in [
    ('Nordic mFRR (DK2)',   params_nordic, bt_nordic, ci_nordic),
    ('German aFRR (DE-LU)', params_german, bt_german, ci_german),
]:
    best = bt.iloc[0]
    print(f'\n{name}:')
    print(f'  β = {params.beta:.3f}')
    print(f'  p^{{b*}} = {best["p_bid"]:.1f} €/MWh  (95% CI: [{ci[0]:.1f}, {ci[1]:.1f}])')
    print(f'  Best OOS strategy: {best["strategy"]} → E[V] = {best["mean_revenue"]:.2f} €/MWh')
    spread = bt.iloc[0]['mean_revenue'] - bt.iloc[-1]['mean_revenue']
    print(f'  Spread (best − worst): {spread:.2f} €/MWh')
print('\n' + '=' * 70)
print('✓ Replication complete')